In [1]:
%env HF_HOME=/data/.cache
import os
os.environ['HF_HOME'] = '/data/.cache'

env: HF_HOME=/data/.cache


In [2]:
import argparse
import logging
import os
from dataclasses import dataclass, field

import torch
import torch.nn.functional as F
from torch.optim import AdamW
from transformers import AutoModelForCausalLM, AutoTokenizer

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
student_model = "Qwen/Qwen3-0.6B"
teacher_model = "Qwen/Qwen3-4B"

Using device: cuda


In [4]:
# ----------------------------------------------------------------
# 載入 tokenizer
# ----------------------------------------------------------------
print(f"Loading tokenizer from {student_model}")
tokenizer = AutoTokenizer.from_pretrained(student_model)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# ----------------------------------------------------------------
# 載入 student model（需要梯度）
# ----------------------------------------------------------------
print(f"Loading student model: {student_model}")
student_model = AutoModelForCausalLM.from_pretrained(
    student_model,
    dtype=torch.bfloat16,
    attn_implementation="flash_attention_2" if device == "cuda" else "eager",
).to(device)

# ----------------------------------------------------------------
# 載入 teacher model（inference only，不需要梯度）
# ----------------------------------------------------------------
print(f"Loading teacher model: {teacher_model}")
teacher_model = AutoModelForCausalLM.from_pretrained(
    teacher_model,
    dtype=torch.bfloat16,
    attn_implementation="flash_attention_2" if device == "cuda" else "eager",
    tie_word_embeddings=False,
).to(device)
teacher_model.eval()
# 凍結 teacher 的所有參數，節省記憶體
for param in teacher_model.parameters():
    param.requires_grad = False
print("Teacher model is freezed")

Loading tokenizer from Qwen/Qwen3-0.6B
Loading student model: Qwen/Qwen3-0.6B


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading teacher model: Qwen/Qwen3-4B


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Qwen3ForCausalLM LOAD REPORT from: Qwen/Qwen3-4B
Key            | Status  | 
---------------+---------+-
lm_head.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Teacher model is freezed


In [5]:
optimizer = AdamW(student_model.parameters(), lr=1e-6)

In [6]:
def get_per_token_logprobs(model, input_ids, attention_mask, target_ids):
    """
    給定一個 model 和一段 token 序列，回傳每個 target token 位置的 log probability。

    Args:
        model: HuggingFace CausalLM
        input_ids: (batch, seq_len) — 完整的輸入序列（prompt + generation）
        attention_mask: (batch, seq_len)
        target_ids: (batch, seq_len) — 要算 logprob 的 target token
                    通常 = input_ids shifted left by 1

    Returns:
        logprobs: (batch, seq_len-1) — 每個 position 對應 target token 的 log prob
    """
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    # logits shape: (batch, seq_len, vocab_size)
    # 我們要的是 position i 預測 position i+1 的 token
    logits = outputs.logits[:, :-1, :]  # (batch, seq_len-1, vocab_size)
    targets = target_ids[:, 1:]          # (batch, seq_len-1)

    # 轉成 log probabilities
    log_probs = F.log_softmax(logits, dim=-1)  # (batch, seq_len-1, vocab_size)

    # 取出每個位置對應 target token 的 log prob
    # gather 沿著 vocab_size 維度，根據 targets 的 index 取值
    token_logprobs = log_probs.gather(
        dim=-1, index=targets.unsqueeze(-1)
    ).squeeze(-1)  # (batch, seq_len-1)

    return token_logprobs

In [7]:
def build_generation_mask(prompt_len, total_len, device):
    """
    建立一個 mask，只在 student generation 的部分（非 prompt）為 1。
    我們只對 student 自己生成的 token 計算 loss，prompt 部分不算。

    Args:
        prompt_len: prompt 的 token 數量
        total_len: 整個序列（prompt + generation）的 token 數量

    Returns:
        mask: (total_len - 1,) — 因為 logprobs 比 input_ids 短 1
    """
    mask = torch.zeros(total_len - 1, device=device)
    # logprobs[i] 對應的是 position i 預測 position i+1
    # generation 從 prompt_len 開始，所以 mask 從 prompt_len-1 開始
    # （因為 logprobs 的 index 已經 shift 了 1）
    mask[prompt_len - 1:] = 1.0
    return mask

In [8]:
# ==============================================================
# Step 1: 取 prompts
# ==============================================================
# 每個 prompt 取樣 samples_per_prompt 次（增加 on-policy 覆蓋率）
expanded_prompts = ["Write a hello world in python", "Hello"]

# Tokenize prompts
prompt_encodings = tokenizer(
    expanded_prompts,
    return_tensors="pt",
    padding=True,
    truncation=True,
).to(device)
prompt_lengths = prompt_encodings.attention_mask.sum(dim=1)  # 每個 prompt 的實際長度

In [9]:
prompt_encodings

{'input_ids': tensor([[  7985,    264,  23811,   1879,    304,  10135],
        [  9707, 151643, 151643, 151643, 151643, 151643]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1],
        [1, 0, 0, 0, 0, 0]], device='cuda:0')}

In [10]:
prompt_lengths

tensor([6, 1], device='cuda:0')

In [11]:
# ==============================================================
# Step 2: Student generate（on-policy 取樣）
#   這裡用 no_grad，因為 generate 本身不需要梯度
# ==============================================================
student_model.eval()
with torch.no_grad():
    generated_outputs = student_model.generate(
        **prompt_encodings,
        max_new_tokens=64,
        do_sample=True,
        temperature=0.6,
        top_p=0.95,
        pad_token_id=tokenizer.pad_token_id,
    )
    # generated_outputs shape: (batch * samples_per_prompt, prompt_len + gen_len)

    # 建立 attention mask（padding token 為 0）
    gen_attention_mask = (generated_outputs != tokenizer.pad_token_id).long()



A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


In [12]:
print(gen_attention_mask[0])
print(gen_attention_mask[1])

tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       device='cuda:0')
tensor([1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       device='cuda:0')


In [13]:
# ==============================================================
# Step 3: 記錄 sampling 時 student 的 log probs（no_grad）
#   這是 importance sampling 中的 q(x)
# ==============================================================
with torch.no_grad():
    sampling_logprobs = get_per_token_logprobs(
        student_model,
        generated_outputs,
        gen_attention_mask,
        generated_outputs,
    )

print("length", len(sampling_logprobs))

length 2


In [14]:
sampling_logprobs

tensor([[-1.2000e+01, -1.1312e+01, -2.3047e-01, -1.7500e+00, -2.4844e+00,
         -2.1719e+00, -2.3281e+00, -4.3164e-01, -1.1406e+00, -1.3516e+00,
         -4.1797e-01, -2.1118e-02, -8.3984e-01, -5.1953e-01, -2.3242e-01,
         -4.3555e-01, -7.2266e-02, -6.2109e-01, -5.5859e-01, -2.7161e-03,
         -8.9453e-01, -1.6895e-01, -3.5156e-01, -4.8828e-03, -4.2969e-01,
         -2.9883e-01, -8.1250e-01, -2.0898e-01, -5.2344e-01, -4.5703e-01,
         -1.7578e-02, -2.0386e-02, -1.2500e-01, -1.3867e-01, -3.7354e-02,
         -1.4587e-02, -1.2970e-03, -1.2268e-02, -4.6692e-03, -2.0885e-04,
         -6.7871e-02, -3.9795e-02, -7.6904e-03, -3.0518e-02, -1.7480e-01,
         -4.8633e-01, -2.0312e-01, -3.1055e-01, -1.0938e+00, -1.0859e+00,
         -1.8203e+00, -1.4062e+00, -9.8438e-01, -1.9653e-02, -2.9062e+00,
         -5.5469e-01, -9.8419e-04, -3.5156e-02, -3.6955e-05, -3.0518e-03,
         -7.8516e-01, -6.0938e-01, -1.9824e-01, -2.1250e+00, -1.4062e+00,
         -5.2002e-02, -5.4297e-01, -6.

In [15]:
# ==============================================================
# Step 4: Teacher forward pass → teacher log probs（no_grad）
# ==============================================================
with torch.no_grad():
    teacher_logprobs = get_per_token_logprobs(
        teacher_model,
        generated_outputs,
        gen_attention_mask,
        generated_outputs,
    )

teacher_logprobs

tensor([[-15.3750, -18.2500, -11.5625, -21.0000, -14.3125, -21.3750, -17.0000,
         -13.5000, -16.7500, -13.0000, -20.5000, -16.3750, -14.3750, -21.7500,
         -13.7500, -18.2500, -15.6875, -16.2500, -13.1875, -18.3750, -20.3750,
         -14.6875, -15.6875, -15.0625, -15.0625, -13.4375, -14.8750, -13.3125,
         -13.9375, -14.7500, -12.9375, -11.9375, -10.0000, -11.5625, -16.6250,
         -15.6250, -15.9375, -14.1875, -17.3750, -18.0000, -19.8750, -13.3125,
         -15.3125, -17.8750, -21.1250, -20.5000, -14.6250, -18.2500, -16.2500,
         -14.7500, -18.2500, -13.3750, -16.7500, -13.0625, -12.0625, -12.5625,
         -15.1250, -19.3750, -15.3750, -14.0625, -15.1875, -12.2500, -14.3125,
         -14.5625, -10.1875, -15.5000, -10.1875, -12.8125, -15.5000],
        [-17.7500, -18.0000, -18.0000, -18.0000, -18.0000, -16.5000, -12.0000,
         -17.2500, -16.5000, -15.3125, -14.2500, -17.6250, -18.2500, -13.6875,
         -14.6875, -19.3750, -17.5000, -15.7500, -16.6250, -2

In [16]:
assert len(teacher_logprobs) == len(sampling_logprobs)
assert len(teacher_logprobs[0]) == len(sampling_logprobs[0])

In [17]:
reverse_kl = sampling_logprobs - teacher_logprobs
print(f"reverse_kl[0][0] = {sampling_logprobs[0][0]} - {teacher_logprobs[0][0]} = {sampling_logprobs[0][0] - teacher_logprobs[0][0]}")
advantages = -reverse_kl
print(f"advantages = -reverse_kl; advantages[0][0] = -reverse_kl[0][0] = {-reverse_kl[0][0]}")

reverse_kl[0][0] = -12.0 - -15.375 = 3.375
advantages = -reverse_kl; advantages[0][0] = -reverse_kl[0][0] = -3.375


In [18]:
advantages

tensor([[ -3.3750,  -6.9375, -11.3125, -19.2500, -11.8125, -19.2500, -14.6875,
         -13.0625, -15.6250, -11.6250, -20.1250, -16.3750, -13.5625, -21.2500,
         -13.5000, -17.8750, -15.6250, -15.6250, -12.6250, -18.3750, -19.5000,
         -14.5000, -15.3125, -15.0625, -14.6250, -13.1250, -14.0625, -13.1250,
         -13.4375, -14.3125, -12.9375, -11.9375,  -9.8750, -11.4375, -16.6250,
         -15.6250, -15.9375, -14.1875, -17.3750, -18.0000, -19.7500, -13.2500,
         -15.3125, -17.8750, -21.0000, -20.0000, -14.4375, -18.0000, -15.1250,
         -13.6875, -16.3750, -12.0000, -15.7500, -13.0625,  -9.1250, -12.0000,
         -15.1250, -19.3750, -15.3750, -14.0625, -14.3750, -11.6250, -14.1250,
         -12.4375,  -8.7500, -15.4375,  -9.6250, -12.1875, -15.5000],
        [ -3.0000,  -3.6250,  -3.6250,  -3.6250,  -3.6250, -12.1250,  -3.1875,
         -13.7500, -15.5000, -13.3750, -13.4375, -15.5000, -17.3750, -12.6250,
         -13.1250, -16.2500, -17.5000, -15.3125, -15.2500, -1

In [19]:
gen_masks = torch.stack([
    build_generation_mask(
        prompt_len=prompt_lengths[i].item(),
        total_len=generated_outputs.shape[1],
        device=device,
    )
    for i in range(generated_outputs.shape[0])
])  # (batch, seq_len-1)

In [20]:
gen_masks

tensor([[0., 0., 0., 0., 0., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]],
       device='cuda:0')

In [21]:
advantages = advantages * gen_masks
advantages

tensor([[ -0.0000,  -0.0000,  -0.0000,  -0.0000,  -0.0000, -19.2500, -14.6875,
         -13.0625, -15.6250, -11.6250, -20.1250, -16.3750, -13.5625, -21.2500,
         -13.5000, -17.8750, -15.6250, -15.6250, -12.6250, -18.3750, -19.5000,
         -14.5000, -15.3125, -15.0625, -14.6250, -13.1250, -14.0625, -13.1250,
         -13.4375, -14.3125, -12.9375, -11.9375,  -9.8750, -11.4375, -16.6250,
         -15.6250, -15.9375, -14.1875, -17.3750, -18.0000, -19.7500, -13.2500,
         -15.3125, -17.8750, -21.0000, -20.0000, -14.4375, -18.0000, -15.1250,
         -13.6875, -16.3750, -12.0000, -15.7500, -13.0625,  -9.1250, -12.0000,
         -15.1250, -19.3750, -15.3750, -14.0625, -14.3750, -11.6250, -14.1250,
         -12.4375,  -8.7500, -15.4375,  -9.6250, -12.1875, -15.5000],
        [ -3.0000,  -3.6250,  -3.6250,  -3.6250,  -3.6250, -12.1250,  -3.1875,
         -13.7500, -15.5000, -13.3750, -13.4375, -15.5000, -17.3750, -12.6250,
         -13.1250, -16.2500, -17.5000, -15.3125, -15.2500, -1

## Importance Sampling Loss:

$$
\begin{align}
\rho &= \exp \left( \log p_{\text{target}} - \log p_{\text{sampling}} \right) \\
\mathcal{L} &= - \sum \left( \rho \cdot A \right)
\end{align}
$$

In [22]:
student_model.train()
optimizer.zero_grad()
current_logprobs = get_per_token_logprobs(
    student_model,
    generated_outputs,
    gen_attention_mask,
    generated_outputs,
)

# Importance sampling ratio: π_θ(x) / q(x)
# 因為是在 log space，所以 exp(log π_θ - log q) = π_θ / q
log_ratio = current_logprobs - sampling_logprobs.detach()
ratio = torch.exp(log_ratio)

# Loss = -(ratio * advantage).sum()
# 每個 token 獨立計算，最後 sum
token_losses = ratio * advantages.detach()
loss = -token_losses.sum()

In [23]:
current_logprobs == sampling_logprobs

tensor([[True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True, True, True, True, True, True]],
       device='cuda:0')

In [24]:
loss

tensor(1902.6875, device='cuda:0', grad_fn=<NegBackward0>)